In [ ]:
import os
import pandas as pd
from collections import Counter
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
from torchvision import models, transforms
import matplotlib.pyplot as plt

# --- 1. Config ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

NUM_EPOCHS = 7
WARMUP_EPOCHS = 3
BATCH_SIZE = 64
LR_WARMUP = 1e-3
LR_FINETUNE = 1e-4

CSV_PATH = "/kaggle/input/cattle-breed-classification-dataset/dataset.csv"
IMG_DIRS = [
    "/kaggle/input/cattle-breed-classification-dataset/dataset/dataset/images",
    "/kaggle/input/cattle-breed-classification-dataset/dataset/dataset/yt_images"
]

# --- 2. Load CSV ---
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["sku", "breed"])
print(f"CSV loaded: {len(df)} rows")

# Encode breeds
breeds = sorted(df["breed"].unique())
breed_to_idx = {b: i for i, b in enumerate(breeds)}
idx_to_breed = {i: b for b, i in breed_to_idx.items()}
num_classes = len(breeds)
print(f"Found {num_classes} breeds")

# --- 3. Transforms ---
data_transforms = {
    "train": transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    "val": transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
}

# --- 4. Custom Dataset ---
class CattleDataset(Dataset):
    def __init__(self, dataframe, img_dirs, transform=None):
        self.samples = []
        self.transform = transform

        for _, row in dataframe.iterrows():
            sku, breed = row["sku"], row["breed"]
            breed_idx = breed_to_idx[breed]

            found_images = False
            for img_dir in img_dirs:
                folder_path = os.path.join(img_dir, sku)
                if not os.path.isdir(folder_path):
                    continue

                for file in os.listdir(folder_path):
                    if file.lower().endswith((".jpg", ".jpeg", ".png")) and not file.startswith("._"):
                        img_path = os.path.join(folder_path, file)
                        self.samples.append((img_path, breed_idx))
                        found_images = True

            if not found_images:
                print(f"⚠️ Warning: No images found for SKU {sku}")

        print(f"Loaded {len(self.samples)} images")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            image = Image.open(path).convert("RGB")
        except:
            return self.__getitem__((idx + 1) % len(self.samples))
        if self.transform:
            image = self.transform(image)
        return image, label

# --- 5. Dataset & Splits ---
full_dataset = CattleDataset(df, IMG_DIRS, transform=data_transforms["train"])
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

# apply correct transforms
train_dataset.dataset.transform = data_transforms["train"]
val_dataset.dataset.transform = data_transforms["val"]

# --- Handle class imbalance ---
targets = [label for _, label in train_dataset]
class_counts = Counter(targets)
class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
sample_weights = [class_weights[label] for label in targets]
if len(sample_weights) > 0:
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
else:
    sampler = None

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# --- 6. Model ---
try:
    model = models.resnet18(weights="IMAGENET1K_V1")
except:
    model = models.resnet18(pretrained=True)

num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(num_ftrs, num_classes),
)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.fc.parameters(), lr=LR_WARMUP)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler = torch.cuda.amp.GradScaler()

# --- 7. Training ---
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []
clean_train_accuracies = []
best_val_acc = 0.0
save_path = "best_resnet18.pth"

print("Starting training...")

for epoch in range(NUM_EPOCHS):
    # freeze first
    if epoch == 0:
        for param in model.parameters():
            param.requires_grad = False
        for param in model.fc.parameters():
            param.requires_grad = True

    # unfreeze
    if epoch == WARMUP_EPOCHS:
        for param in model.parameters():
            param.requires_grad = True
        optimizer = optim.Adam(model.parameters(), lr=LR_FINETUNE)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - WARMUP_EPOCHS)
        print("\n Unfroze all layers for fine-tuning\n")

    # --- Train ---
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
    epoch_train_loss = running_loss / total_train
    epoch_train_acc = 100.0 * correct_train / total_train
    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_acc)

    # --- Validation ---
    model.eval()
    running_val_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = 100.0 * correct_val / total_val
    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_acc)

    # --- Clean train acc ---
    clean_correct, clean_total = 0, 0
    clean_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    with torch.no_grad():
        for images, labels in clean_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            clean_total += labels.size(0)
            clean_correct += (predicted == labels).sum().item()
    clean_train_acc = 100.0 * clean_correct / clean_total
    clean_train_accuracies.append(clean_train_acc)

    scheduler.step()

    # save best
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), save_path)
        print(f"Saved best model at epoch {epoch+1} with Val Acc: {epoch_val_acc:.2f}%")

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
          f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}% | "
          f"Clean Train Acc: {clean_train_acc:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%")

# save final
torch.save(model.state_dict(), "final_resnet18.pth")
print("Training finished. Final model saved.")

# --- 8. Plot ---
plt.figure(figsize=(10,5))
plt.plot(train_accuracies, label="Train Acc (augmented)")
plt.plot(clean_train_accuracies, label="Clean Train Acc")
plt.plot(val_accuracies, label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import numpy as np
import time
from sklearn.preprocessing import label_binarize
from itertools import cycle
#-# --- 9. Evaluation ---
print("\n\n=== EVALUATION ON FINAL MODEL ===")

# Load final model
final_model = models.resnet18(weights=None)
num_ftrs = final_model.fc.in_features
final_model.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(num_ftrs, num_classes),
)
final_model.load_state_dict(torch.load("final_resnet18.pth", map_location=device))
final_model = final_model.to(device)
final_model.eval()

# Collect predictions
all_labels, all_preds, all_probs = [], [], []
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = final_model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=breeds)
fig, ax = plt.subplots(figsize=(10,10))
disp.plot(ax=ax, cmap="Blues", xticks_rotation=90)
plt.title("Confusion Matrix")
plt.show()

# Classification Report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=breeds))

# ROC Curve & AUC
y_bin = label_binarize(all_labels, classes=list(range(num_classes)))
fpr, tpr, roc_auc = dict(), dict(), dict()
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(8,6))
for i, color in zip(range(num_classes), cycle(plt.cm.tab20.colors)):
    plt.plot(fpr[i], tpr[i], color=color, lw=1.5, 
             label=f"{breeds[i]} (AUC={roc_auc[i]:.2f})")
plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (One-vs-Rest)")
plt.legend(fontsize=8)
plt.show()

# Model Info
total_params = sum(p.numel() for p in final_model.parameters())
trainable_params = sum(p.numel() for p in final_model.parameters() if p.requires_grad)
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Model Size: {os.path.getsize('final_resnet18.pth')/1e6:.2f} MB")

# Inference Speed Test
dummy_input = torch.randn(1, 3, 224, 224).to(device)
start = time.time()
with torch.no_grad():
    for _ in range(100):
        _ = final_model(dummy_input)
end = time.time()
print(f"Inference Speed: {(end-start)/100:.4f} sec per image")